# M3L2 E07 - Retriever: la interfaz de busqueda

## Un solo concepto

En E06 usamos `vectorstore.similarity_search(query, k=2)` directamente.

Eso funciona, pero tiene un problema: si queremos cambiar de FAISS a Pinecone o Chroma,
hay que cambiar todas las llamadas a `similarity_search` en el codigo.

**El Retriever** es una capa de abstraccion que expone UNA interfaz estandar:

```python
retriever.invoke(query)  # siempre devuelve List[Document]
```

Sin importar si internamente usa FAISS, Chroma, Pinecone o cualquier otra cosa.

La lecture (Seccion 13.3) dice:

> "Si el retrieval esta encapsulado, puedes cambiar la tecnologia sin romper el resto."

## Necesita OpenAI API key y faiss-cpu


In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()

DOCS = [
    "La politica de vacaciones es de 15 dias por ano.",
    "El seguro medico esta incluido desde el primer dia.",
    "El horario de trabajo es de 9 a 18 con almuerzo.",
    "El trabajo remoto esta permitido 3 dias por semana.",
    "Los bonos anuales se pagan en diciembre.",
]

vectorstore = FAISS.from_texts(DOCS, embeddings)
print("Vector store listo.")


## El Retriever: la tool de busqueda del agente (Lecture M3L2 - Seccion 13.2)

La lecture M3L2 describe el retriever asi:

> "Un Retriever es la interfaz que recupera documentos relevantes."
> -- Lecture M3L2, Seccion 13.2

**Conexion con M3L1**: en el agente de M3L1, si habiamos una tool de busqueda era algo como:

```python
def buscar_documentos(query: str) -> list:
    # busqueda naive: comparar strings
    return [doc for doc in DOCS if query.lower() in doc.lower()]
```

El retriever de LangChain hace lo mismo, pero:
- usa similitud vectorial en lugar de comparar strings
- es intercambiable (FAISS, Chroma, Pinecone)
- tiene la misma interfaz `.invoke()` que cualquier otro componente

**Por que la interfaz estandar importa** (Lecture M3L2 - Seccion 13.3):

```text
Con similarity_search directo:        Con as_retriever():
---------------------------------     ----------------------------------
vectorstore.similarity_search(q,k=2) retriever.invoke(q)

Si cambias de FAISS a Chroma:         Si cambias de FAISS a Chroma:
  - hay que cambiar TODAS las           - cambias UNA linea:
    llamadas a similarity_search         vectorstore = Chroma(...)
                                       - el resto del pipeline NO cambia
```

**En el pipeline RAG completo** (E02), el retriever se conecta asi:

```text
Consulta
   |
   v
retriever.invoke(consulta)  -> [Doc1, Doc2]  <- busqueda vectorial
   |
   v
format_docs([Doc1, Doc2])   -> "Doc1\n\nDoc2"  <- texto para el prompt
   |
   v
ChatPromptTemplate({context: ..., question: ...})
   |
   v
ChatOpenAI
   |
   v
StrOutputParser -> respuesta final
```


## Diferencia: similarity_search vs as_retriever

Dos formas de buscar. La diferencia es si el resultado esta "atado" a FAISS o no.


In [ ]:
# Forma 1: similarity_search directo
# Especifico de FAISS: si cambias de vector store, hay que cambiar este codigo
docs_direct = vectorstore.similarity_search("vacaciones", k=2)
print("Forma 1 (similarity_search):")
print(f"  Tipo devuelto: {type(docs_direct).__name__}")
for doc in docs_direct:
    print(f"  - {doc.page_content}")
print()


## TODO 1: crear el retriever con as_retriever

El retriever se crea desde el vector store con `.as_retriever()`.

El parametro `search_kwargs={"k": N}` controla cuantos documentos traer.


In [ ]:
# TODO 1: crear el retriever con vectorstore.as_retriever(search_kwargs={"k": 2})
retriever = None  # reemplazar

print(f"Tipo del retriever: {type(retriever).__name__ if retriever else 'TODO no completado'}")


## TODO 2: invocar el retriever con .invoke()

El retriever usa `.invoke(query)` en vez de `.similarity_search(query, k=N)`.
El resultado es el mismo (lista de Documents), pero la interfaz es estandar.


In [ ]:
# TODO 2: invocar el retriever con retriever.invoke("vacaciones")
# docs_retriever = retriever.invoke("vacaciones")

# Descomentar:
# print("Forma 2 (as_retriever + invoke):")
# print(f"  Tipo devuelto: {type(docs_retriever).__name__}")
# for doc in docs_retriever:
#     print(f"  - {doc.page_content}")
# print()
# print("Mismo resultado, interfaz estandar.")
# print("Si cambias FAISS por Chroma, solo cambia la linea de 'vectorstore ='")
# print("El retriever.invoke() sigue funcionando igual.")


## TODO 3: experimentar con distintos valores de k

El parametro `k` controla cuantos documentos trae el retriever.
Mas documentos = mas contexto pero mas tokens = mayor costo.


In [ ]:
# TODO 3: crear dos retrievers con k=1 y k=3, invocar con la misma query y comparar
# retriever_k1 = vectorstore.as_retriever(search_kwargs={"k": 1})
# retriever_k3 = vectorstore.as_retriever(search_kwargs={"k": 3})

# query = "politica de empresa"
# print(f"k=1: {len(retriever_k1.invoke(query))} doc(s)")
# print(f"k=3: {len(retriever_k3.invoke(query))} doc(s)")
# print()
# print("Tradeoff: k mas alto = mas contexto pero mas tokens y posiblemente mas ruido")


In [ ]:
def run_checks():
    from langchain_core.vectorstores import VectorStoreRetriever
    assert retriever is not None, "TODO 1: retriever es None"
    # El retriever devuelve una lista de Documents
    docs = retriever.invoke("vacaciones")
    assert isinstance(docs, list) and len(docs) > 0
    assert hasattr(docs[0], 'page_content')
    assert len(docs) <= 2, "Con k=2 no debe devolver mas de 2 docs"
    # El retriever es selectivo: para 'remoto' trae el doc de remoto
    docs_remoto = retriever.invoke("trabajo desde casa")
    contenidos = [d.page_content for d in docs_remoto]
    assert any("remoto" in c.lower() for c in contenidos)
    print("M3L2 E07 checks passed")

run_checks()


## Cierre

| `similarity_search` | `as_retriever` |
|---|---|
| Especifico de FAISS | Interfaz estandar LangChain |
| `vectorstore.similarity_search(q, k=2)` | `retriever.invoke(q)` |
| Cambiar de FAISS = cambiar TODAS las llamadas | Cambiar de FAISS = cambiar solo la linea del vectorstore |
| No se puede pasar a una chain LCEL | Se conecta directamente con `\|` en una chain |

**El retriever se conecta asi en E02:**

```python
{"context": retriever | format_docs, "question": RunnablePassthrough()}
```

El retriever busca los docs, `format_docs` los convierte en texto, y ese texto va al prompt.
